In [1]:
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.stats import spearmanr
from tqdm.auto import tqdm

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if all((p / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

# Run from `code/` and expose both `code/` and `code/kev/` on sys.path
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from data.base import (
    build_long_records,
    fused_eval_context,
    group_records_by_state,
    make_grouped_view,
    make_view,
    predict_probs_grouped,
)
from data.wu2025 import WU2025_PERSONA_COLS, load_wu2025_df
from data.wu2025.stimulus import ATTR_LOCAL, ATTR_MONEY, ATTR_PERCENT, CONJOINT_OPTIONS
from models.kev_pointer import KevPointerModel
from training import free_gpu

print(f"CODE_DIR: {CODE_DIR}")

CODE_DIR: /home/dw/github/decision-models-finetuning/code


In [24]:
# Load all included Wu 2025 survey rows and filter to paired electricity-plan conjoint tasks (`plan_choice`)
t_load0 = time.perf_counter()
wu2025_df = load_wu2025_df()
plan_choice_df = wu2025_df[wu2025_df["v"] == "plan_choice"].reset_index(drop=True)
load_wu_sec = time.perf_counter() - t_load0
if len(plan_choice_df) == 0:
    raise ValueError("No plan_choice rows found in wu2025_df")

# Summarize dataset size and load time: each answered task contributes 2 option rows to OLS
display(
    pd.Series(
        {
            "wu2025_total_rows": len(wu2025_df),
            "wu2025_questions": wu2025_df["v"].nunique(),
            "plan_choice_tasks": len(plan_choice_df),
            "plan_choice_option_rows": len(plan_choice_df) * 2,
            "respondents_with_plan_choice": plan_choice_df["CASEID"].nunique(),
            "distinct_displayed_pairs": plan_choice_df["qid"].nunique(),
            "load_sec": round(load_wu_sec, 4),
        },
        name="wu2025_overview",
    ).to_frame()
)

,wu2025_overview
wu2025_total_rows,90979.0000
wu2025_questions,46.0000
plan_choice_tasks,18591.0000
plan_choice_option_rows,37182.0000
respondents_with_plan_choice,1861.0000
distinct_displayed_pairs,8212.0000
load_sec,0.1978


In [25]:
# Ordered factor levels per attribute; index 0 in each list is the OLS reference level ($105, 10%, No)
WU2025_LEVELS = {
    ATTR_MONEY: ["$105", "$110", "$115", "$120", "$125"],
    ATTR_PERCENT: ["10%", "30%", "50%", "100%"],
    ATTR_LOCAL: ["No", "Yes"],
}

# Ordered names for the 8 non-intercept AMCE coefficients matching columns 1..8 of X
WU2025_BETA_NAMES = [
    *(f"{ATTR_MONEY}: {lvl} vs $105" for lvl in WU2025_LEVELS[ATTR_MONEY][1:]),
    *(f"{ATTR_PERCENT}: {lvl} vs 10%" for lvl in WU2025_LEVELS[ATTR_PERCENT][1:]),
    *(f"{ATTR_LOCAL}: {lvl} vs No" for lvl in WU2025_LEVELS[ATTR_LOCAL][1:]),
]

# SVD relative singular-value cutoff used to verify full column rank before inverting X
OLS_SVD_RCOND = 1e-8


def parse_wu2025_plan_description(desc: str) -> dict[str, str]:
    # Attribute display order is randomized per respondent, so split into 3 `Attribute: Level` clauses
    parts = desc.split("; ")
    if len(parts) != 3:
        raise ValueError(f"Expected 3 attribute clauses, got {desc!r}")
    kv: dict[str, str] = {}
    for part in parts:
        attr, sep, level = part.partition(": ")
        if not sep:
            raise ValueError(f"Malformed attribute clause {part!r} in {desc!r}")
        # Fail fast if any attribute name or level deviates from the known experimental design
        if attr not in WU2025_LEVELS:
            raise KeyError(f"Unexpected attribute {attr!r} in {desc!r}")
        if level not in WU2025_LEVELS[attr]:
            raise ValueError(f"Unexpected level {level!r} for {attr!r}")
        kv[attr] = level
    if set(kv.keys()) != set(WU2025_LEVELS.keys()):
        raise ValueError(f"Missing attributes in {desc!r}: got {sorted(kv.keys())}")
    return kv


def encode_wu2025_x_row(desc: str) -> list[float]:
    # Convert one option's description into [1.0 (intercept), 4 bill dummies, 3 renewables-% dummies, 1 local dummy]
    parsed = parse_wu2025_plan_description(desc)
    return [
        1.0,  # col 0: constant intercept (absorbs baseline choice level, dropped after projection)
        *[1.0 if parsed[ATTR_MONEY] == lvl else 0.0 for lvl in WU2025_LEVELS[ATTR_MONEY][1:]],      # cols 1..4: $110, $115, $120, $125
        *[1.0 if parsed[ATTR_PERCENT] == lvl else 0.0 for lvl in WU2025_LEVELS[ATTR_PERCENT][1:]],  # cols 5..7: 30%, 50%, 100%
        *[1.0 if parsed[ATTR_LOCAL] == lvl else 0.0 for lvl in WU2025_LEVELS[ATTR_LOCAL][1:]],      # col 8: Yes
    ]


def build_ols_projector(X: torch.Tensor) -> torch.Tensor:
    # Build fixed linear OLS operator A of shape (K, N) mapping N option outcomes -> K non-intercept betas
    if X.ndim != 2 or X.shape[0] < X.shape[1] or X.shape[1] < 2:
        raise ValueError(f"Expected 2D full-column design matrix X with N >= P >= 2, got shape {tuple(X.shape)}")
    X64 = X.to(dtype=torch.float64)
    # Fail fast via SVD if X is rank-deficient
    svals = torch.linalg.svdvals(X64)
    if float(svals[-1]) <= OLS_SVD_RCOND * float(svals[0]):
        raise ValueError(f"Rank-deficient design matrix X: min_sval={float(svals[-1]):.3e}, max_sval={float(svals[0]):.3e}")
    # pinv(X64) computes (X^T X)^{-1} X^T via SVD without squaring the condition number; [1:] drops the intercept
    return torch.linalg.pinv(X64)[1:]


def fit_ols_betas(A: torch.Tensor, outcome: torch.Tensor) -> torch.Tensor:
    # Linear OLS projection: beta = A @ outcome (differentiable whenever `outcome` comes from model predictions)
    if outcome.ndim != 1 or outcome.shape[0] != A.shape[1]:
        raise ValueError(f"Expected 1D outcome of length {A.shape[1]}, got shape {tuple(outcome.shape)}")
    return A @ outcome.to(dtype=A.dtype)


def build_ols_inputs(
    tasks_df: pd.DataFrame,
    device: str,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # Validate every 2-option task row and build stacked (2*T, 9) X, (2*T,) targets y, and (8, 2*T) projector A
    if len(tasks_df) == 0:
        raise ValueError("Cannot build OLS inputs from an empty DataFrame")
    x_rows: list[list[float]] = []
    y_vals: list[float] = []
    for task_idx, row in enumerate(tasks_df.itertuples(index=False)):
        if list(row.option_texts) != CONJOINT_OPTIONS:
            raise ValueError(f"Task {task_idx}: expected options {CONJOINT_OPTIONS}, got {row.option_texts}")
        if row.option_descriptions is None or len(row.option_descriptions) != 2:
            raise ValueError(f"Task {task_idx}: expected 2 option_descriptions, got {row.option_descriptions}")
        one_hot = [float(v) for v in row.one_hot]
        if len(one_hot) != 2 or sorted(one_hot) != [0.0, 1.0]:
            raise ValueError(f"Task {task_idx}: expected binary one_hot of length 2, got {one_hot}")
        for desc, y_val in zip(row.option_descriptions, one_hot, strict=True):
            x_rows.append(encode_wu2025_x_row(desc))
            y_vals.append(y_val)
    X_mat = torch.tensor(x_rows, dtype=torch.float64)
    y_flat = torch.tensor(y_vals, dtype=torch.float64, device=device)
    A_mat = build_ols_projector(X_mat).to(device)
    return X_mat, y_flat, A_mat


print(f"Configured {len(WU2025_BETA_NAMES)} non-intercept OLS beta terms (+ 1 intercept = {len(WU2025_BETA_NAMES) + 1} design columns)")

Configured 8 non-intercept OLS beta terms (+ 1 intercept = 9 design columns)


In [26]:
def compute_wu2025_loss(
    task_logits: torch.Tensor,
    y_flat: torch.Tensor,
    A: torch.Tensor,
    beta_lambda: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # Validate paired 2-option task logits against flattened option targets and precomputed OLS projector A
    if task_logits.ndim != 2 or task_logits.shape[1] != 2:
        raise ValueError(f"Expected (num_tasks, 2) task_logits, got shape {tuple(task_logits.shape)}")
    num_rows = task_logits.numel()
    if y_flat.shape != (num_rows,) or A.shape[1] != num_rows:
        raise ValueError(f"Shape mismatch: num_rows={num_rows}, y_flat={tuple(y_flat.shape)}, A={tuple(A.shape)}")
    if y_flat.requires_grad or A.requires_grad:
        raise ValueError("Human choice targets `y_flat` and OLS projector `A` must be fixed (requires_grad=False)")

    # 1. Per-response KL divergence KL(y || p) averaged across the respondent's choice tasks (in nats)
    log_probs = torch.log_softmax(task_logits, dim=-1)
    y_2col = y_flat.view(-1, 2)
    kl_loss = torch.nn.functional.kl_div(log_probs, y_2col, reduction="batchmean")

    # 2. When beta_lambda == 0, compute raw beta MSE under no_grad for logging so backward() stays pure KL
    if beta_lambda == 0.0:
        with torch.no_grad():
            pred_probs_detached = torch.softmax(task_logits.detach(), dim=-1).reshape(-1)
            beta_loss = (fit_ols_betas(A, pred_probs_detached) - fit_ols_betas(A, y_flat)).pow(2).mean()
        return kl_loss, kl_loss, beta_loss

    # 3. Differentiable OLS beta MSE loss: mean((A @ pred_probs - A @ y_flat)^2) across the K=8 attribute terms
    pred_probs = torch.softmax(task_logits, dim=-1).reshape(-1)
    pred_betas = fit_ols_betas(A, pred_probs)
    human_betas = fit_ols_betas(A, y_flat)
    beta_loss = (pred_betas - human_betas).pow(2).mean()

    # 4. Combined single-pass objective: choice KL loss + beta_lambda * regression-beta MSE
    total_loss = kl_loss + beta_lambda * beta_loss
    return total_loss, kl_loss, beta_loss


print("Defined compute_wu2025_loss(task_logits, y_flat, A, beta_lambda)")

Defined compute_wu2025_loss(task_logits, y_flat, A, beta_lambda)


In [27]:
# Require CUDA and clear any cached allocator blocks before loading the 800M backbone
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required to run jaredpalmer/kev-0.8b in bf16 train mode")
# Release any previously loaded `kev_model` in an active kernel before allocating a fresh 800M backbone
globals().pop("kev_model", None)
device = "cuda"
free_gpu()
torch.manual_seed(42)

# Build `jaredpalmer/kev-0.8b` with the v5 training configuration (bf16 frozen backbone + rank-64 LoRA + pointer head)
t_model0 = time.perf_counter()
kev_model = KevPointerModel.build(
    "jaredpalmer/kev-0.8b",
    device=device,
    lora_rank=64,
    weights_dtype="bf16",
)
kev_model.net.head.temperature = 1.0  # training uses raw uncalibrated temperature = 1.0
kev_model.train()                     # enable training mode (LoRA dropout active, trainable params require_grad)
torch.cuda.synchronize()
model_load_sec = time.perf_counter() - t_model0

# Summarize loaded model state, parameter counts, and GPU load time
trainable_params_count = sum(p.numel() for p in kev_model.trainable_parameters())
total_params_count = sum(p.numel() for p in kev_model.parameters())
display(
    pd.Series(
        {
            "base_model": "jaredpalmer/kev-0.8b",
            "mode": kev_model.mode,
            "training": kev_model.training,
            "weights_dtype": kev_model.meta.weights_dtype,
            "lora_rank": kev_model.meta.lora,
            "trainable_params": f"{trainable_params_count:,}",
            "total_params": f"{total_params_count:,}",
            "model_load_sec": round(model_load_sec, 3),
        },
        name="kev_800m_train_mode",
    ).to_frame()
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

,kev_800m_train_mode
base_model,jaredpalmer/kev-0.8b
mode,lora_rank 64
training,True
weights_dtype,bf16
lora_rank,64
trainable_params,"43,815,424"
total_params,"796,208,448"
model_load_sec,1.461


In [28]:
# Build state-grouped evaluation bundle across all 18,591 Wu 2025 `plan_choice` tasks (1,831 unique persona states)
t_pop0 = time.perf_counter()
wu2025_pop_records = build_long_records(
    plan_choice_df,
    persona_cols=WU2025_PERSONA_COLS,
    include_country=False,
    year="wu2025",
)
wu2025_pop_bundle = group_records_by_state(
    wu2025_pop_records,
    view_fn=make_view,
    name="wu2025_plan_choice",
    valid=True,
)

# Precompute fixed population OLS projector `A_pop` (8 x 37,182) and human population target betas `human_pop_betas`
X_pop, y_pop, A_pop = build_ols_inputs(plan_choice_df, device="cpu")
human_pop_betas = fit_ols_betas(A_pop, y_pop)
pop_bundle_prep_sec = time.perf_counter() - t_pop0
if bool((human_pop_betas == 0).any()):
    raise ValueError("Zero population human OLS beta encountered; MAPE denominator would be undefined")

print(
    f"Population eval bundle: {len(wu2025_pop_bundle['records']):,} tasks | "
    f"{len(wu2025_pop_bundle['packed_views']):,} packed persona states | "
    f"A_pop shape: {tuple(A_pop.shape)} | prep_sec: {pop_bundle_prep_sec:.3f}s"
)
display(pd.Series(human_pop_betas.numpy(), index=WU2025_BETA_NAMES, name="human_pop_beta").round(4).to_frame())

Population eval bundle: 18,591 tasks | 1,831 packed persona states | A_pop shape: (8, 37182) | prep_sec: 0.719s


,human_pop_beta
Monthly bill: $110 vs $105,-0.0969
Monthly bill: $115 vs $105,-0.1779
Monthly bill: $120 vs $105,-0.2894
Monthly bill: $125 vs $105,-0.3832
Percentage of renewables: 30% vs 10%,0.0973
Percentage of renewables: 50% vs 10%,0.1706
Percentage of renewables: 100% vs 10%,0.2421
Locally produced renewables: Yes vs No,0.1042


In [29]:
@torch.no_grad()
def validate_wu2025_population_ols(
    model: KevPointerModel,
    pop_bundle: dict,
    A_pop: torch.Tensor,
    y_pop: torch.Tensor,
    human_pop_betas: torch.Tensor,
) -> tuple[dict[str, float], pd.DataFrame]:
    # Time full evaluation wall-clock (including LoRA fusion / CudaGraphs setup + GPU inference + OLS projection)
    torch.cuda.synchronize()
    t_eval_wall0 = time.perf_counter()

    # Switch to eval mode and fuse current LoRA deltas + CudaGraphs for shared-state inference across all respondents
    was_training = model.training
    model.eval()
    free_gpu()
    try:
        with torch.autocast("cuda", dtype=torch.bfloat16), fused_eval_context(model, eval_dtype=torch.bfloat16):
            preds, stats = predict_probs_grouped(model, pop_bundle, show_progress=False, return_stats=True)
    finally:
        # Always restore training mode and release temporary graph buffers so training can resume immediately
        if was_training:
            model.train()
        free_gpu()

    # Fail fast if fused_eval_context failed to restore trainable parameter state or training mode
    if model.training != was_training:
        raise AssertionError(f"Expected model.training={was_training} after validation, got {model.training}")
    if any((not p.requires_grad) or p.numel() == 0 for p in model.trainable_parameters()):
        raise AssertionError("Trainable parameters were not cleanly restored after fused_eval_context")

    # Time CPU tensor stacking, population KL computation, and `A_pop @ pred_probs` OLS projection
    t_ols0 = time.perf_counter()
    num_tasks = y_pop.numel() // 2
    if len(preds) != num_tasks:
        raise ValueError(f"Expected {num_tasks} task predictions, got {len(preds)}")
    pred_probs_2col = torch.from_numpy(np.stack(preds, axis=0)).to(dtype=torch.float64)
    if pred_probs_2col.shape != (num_tasks, 2):
        raise ValueError(f"Expected ({num_tasks}, 2) predicted probabilities, got {tuple(pred_probs_2col.shape)}")

    pop_kl_loss = float(
        torch.nn.functional.kl_div(
            pred_probs_2col.clamp_min(1e-12).log(),
            y_pop.view(-1, 2),
            reduction="batchmean",
        )
    )
    model_pop_betas = fit_ols_betas(A_pop, pred_probs_2col.reshape(-1))
    beta_diff = model_pop_betas - human_pop_betas
    eval_ols_ms = (time.perf_counter() - t_ols0) * 1000.0
    eval_wall_sec = time.perf_counter() - t_eval_wall0

    # Report statistical OLS / KL metrics side-by-side with detailed evaluation timing metrics
    metrics = {
        "pop_kl_loss": pop_kl_loss,
        "pop_beta_mse": float(beta_diff.pow(2).mean()),
        "pop_beta_mae": float(beta_diff.abs().mean()),
        "pop_beta_mape_pct": float((beta_diff.abs() / human_pop_betas.abs()).mean() * 100.0),
        "spearman_rho": float(spearmanr(model_pop_betas.numpy(), human_pop_betas.numpy()).statistic),
        "eval_prep_sec": float(stats["prep_sec"]),
        "eval_gpu_sec": float(stats["gpu_sec"]),
        "eval_ols_ms": eval_ols_ms,
        "eval_wall_sec": eval_wall_sec,
        "eval_ms_per_task": (eval_wall_sec * 1000.0) / num_tasks,
    }
    beta_df = pd.DataFrame(
        {
            "human_pop_beta": human_pop_betas.numpy(),
            "model_pop_beta": model_pop_betas.numpy(),
            "diff": beta_diff.numpy(),
            "abs_diff": beta_diff.abs().numpy(),
        },
        index=pd.Index(WU2025_BETA_NAMES, name="term"),
    )
    return metrics, beta_df


# Run Step 0 baseline validation on untrained `jaredpalmer/kev-0.8b` before any training updates
step0_val_metrics, step0_beta_df = validate_wu2025_population_ols(
    kev_model,
    wu2025_pop_bundle,
    A_pop,
    y_pop,
    human_pop_betas,
)
display(pd.Series(step0_val_metrics, name="step_0_population_val").round(4).to_frame())
display(step0_beta_df.round(4))

,step_0_population_val
pop_kl_loss,1.0699
pop_beta_mse,0.0512
pop_beta_mae,0.2093
pop_beta_mape_pct,114.1314
spearman_rho,-0.2143
eval_prep_sec,0.1943
eval_gpu_sec,32.9060
eval_ols_ms,5.7037
eval_wall_sec,34.2062
eval_ms_per_task,1.8399


,human_pop_beta,model_pop_beta,diff,abs_diff
term,,,,
Monthly bill: $110 vs $105,-0.0969,-0.0068,0.0901,0.0901
Monthly bill: $115 vs $105,-0.1779,-0.0122,0.1658,0.1658
Monthly bill: $120 vs $105,-0.2894,-0.0171,0.2723,0.2723
Monthly bill: $125 vs $105,-0.3832,-0.0248,0.3584,0.3584
Percentage of renewables: 30% vs 10%,0.0973,-0.0075,-0.1047,0.1047
Percentage of renewables: 50% vs 10%,0.1706,-0.0149,-0.1856,0.1856
Percentage of renewables: 100% vs 10%,0.2421,-0.0398,-0.2820,0.2820
Locally produced renewables: Yes vs No,0.1042,-0.1109,-0.2151,0.2151


In [30]:
# Snapshot all 376 untrained LoRA + pointer-head tensors on CPU so every `beta_lambda` run starts identically
INIT_TRAINABLE_STATE = {
    name: param.detach().cpu().clone()
    for name, param in kev_model.named_parameters()
    if param.requires_grad
}


def reset_kev_trainable_weights(model: KevPointerModel, init_state: dict[str, torch.Tensor]) -> None:
    # Fail fast if the model's trainable parameter set changed before copying Step-0 weights back onto GPU
    current_names = {name for name, param in model.named_parameters() if param.requires_grad}
    if current_names != set(init_state.keys()):
        raise ValueError("Trainable parameter names do not match INIT_TRAINABLE_STATE snapshot")
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.copy_(init_state[name].to(device=param.device, dtype=param.dtype))
                param.grad = None
    model.train()
    free_gpu()


# Pre-tokenize all 1,861 respondents once and precompute per-respondent `A_resp` where `X_resp` has full column rank 9/9
t_resp_prep0 = time.perf_counter()
wu2025_respondent_items: list[dict] = []
full_rank_count = 0
rank_deficient_count = 0

for r_caseid, r_group in tqdm(
    plan_choice_df.groupby("CASEID", sort=False),
    total=plan_choice_df["CASEID"].nunique(),
    desc="Preparing respondents",
):
    r_df = r_group.reset_index(drop=True)
    x_rows: list[list[float]] = []
    y_vals: list[float] = []
    for t_idx, row in enumerate(r_df.itertuples(index=False)):
        # Validate every respondent task's options, descriptions, and binary one-hot choice vector
        if list(row.option_texts) != CONJOINT_OPTIONS:
            raise ValueError(f"{r_caseid} task {t_idx}: expected {CONJOINT_OPTIONS}, got {row.option_texts}")
        if row.option_descriptions is None or len(row.option_descriptions) != 2:
            raise ValueError(f"{r_caseid} task {t_idx}: expected 2 option_descriptions")
        one_hot = [float(v) for v in row.one_hot]
        if len(one_hot) != 2 or sorted(one_hot) != [0.0, 1.0]:
            raise ValueError(f"{r_caseid} task {t_idx}: invalid binary one_hot {one_hot}")
        for desc, y_val in zip(row.option_descriptions, one_hot, strict=True):
            x_rows.append(encode_wu2025_x_row(desc))
            y_vals.append(y_val)

    # Tokenize the respondent's shared persona state + choice tasks once and verify target alignment
    r_records = build_long_records(r_df, persona_cols=WU2025_PERSONA_COLS, include_country=False, year="wu2025")
    r_enc, r_targets = kev_model.prepare(make_grouped_view(r_records))
    r_y = torch.tensor(y_vals, dtype=torch.float64, device=device)
    prepared_2col = torch.stack([q["target"].to(dtype=torch.float64, device=device) for q in r_targets], dim=0)
    if not torch.equal(prepared_2col, r_y.view(-1, 2)):
        raise AssertionError(f"{r_caseid}: Kev prepared target order does not match y_resp")

    # Check SVD column rank explicitly: 1,725 respondents have full rank 9/9; 136 have <9 rows or miss a level / are collinear
    r_X = torch.tensor(x_rows, dtype=torch.float64)
    if r_X.shape[0] >= r_X.shape[1]:
        r_svals = torch.linalg.svdvals(r_X)
        is_full_rank = float(r_svals[-1]) > OLS_SVD_RCOND * float(r_svals[0])
    else:
        is_full_rank = False

    if is_full_rank:
        r_A = torch.linalg.pinv(r_X)[1:].to(device=device)
        full_rank_count += 1
    else:
        r_A = None  # standalone 8-beta OLS is under-identified for this respondent; trained with KL loss only
        rank_deficient_count += 1

    wu2025_respondent_items.append(
        {
            "caseid": str(r_caseid),
            "num_tasks": len(r_df),
            "enc": r_enc,
            "y_resp": r_y,
            "A_resp": r_A,
        }
    )

resp_prep_sec = time.perf_counter() - t_resp_prep0
if (len(wu2025_respondent_items), full_rank_count, rank_deficient_count) != (1861, 1725, 136):
    raise AssertionError(
        f"Unexpected respondent counts: {(len(wu2025_respondent_items), full_rank_count, rank_deficient_count)}"
    )

# Registry storing each `beta_lambda` run's full step/timing logs and population OLS trajectory
lambda_runs: dict[float, dict] = {}

display(
    pd.Series(
        {
            "total_respondents": len(wu2025_respondent_items),
            "full_rank_ols_respondents": full_rank_count,
            "rank_deficient_kl_only_respondents": rank_deficient_count,
            "snapshot_trainable_tensors": len(INIT_TRAINABLE_STATE),
            "respondent_prep_sec": round(resp_prep_sec, 3),
            "ms_per_respondent_prep": round((resp_prep_sec * 1000.0) / len(wu2025_respondent_items), 3),
        },
        name="epoch_prep_summary",
    ).to_frame()
)

Preparing respondents:   0%|          | 0/1861 [00:00<?, ?it/s]

,epoch_prep_summary
total_respondents,1861.000
full_rank_ols_respondents,1725.000
rank_deficient_kl_only_respondents,136.000
snapshot_trainable_tensors,376.000
respondent_prep_sec,7.831
ms_per_respondent_prep,4.208


In [31]:
# Shared single-respondent (`grad_accum=1`) hyperparameters held identical across all `beta_lambda` runs
EPOCH_SEED = 42
EPOCH_LR = 2e-5            # scaled for 1-respondent steps (`grad_accum=1`, 1,861 steps/epoch)
EPOCH_WEIGHT_DECAY = 0.01
EPOCH_VAL_EVERY = 450      # validate population OLS at steps 0, 450, 900, 1350, and 1861


def run_wu2025_epoch(beta_lambda: float) -> dict:
    t_wall0 = time.perf_counter()

    # Reset weights to exact Step-0 initialization and re-seed RNGs so respondent order & dropout match across runs
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)
    rng = random.Random(EPOCH_SEED)
    torch.manual_seed(EPOCH_SEED)
    torch.cuda.manual_seed_all(EPOCH_SEED)

    epoch_items = list(wu2025_respondent_items)
    rng.shuffle(epoch_items)
    total_steps = len(epoch_items)

    # Configure AdamW + OneCycleLR (`grad_accum=1`: 1 optimizer update per respondent)
    trainable_params = list(kev_model.trainable_parameters())
    optimizer = torch.optim.AdamW(trainable_params, lr=EPOCH_LR, weight_decay=EPOCH_WEIGHT_DECAY, fused=True)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=EPOCH_LR,
        total_steps=total_steps,
        pct_start=0.1,
        div_factor=25.0,
        final_div_factor=1e4,
    )

    step_rows: list[dict] = []
    val_rows: list[dict] = [{"step": 0, "train_elapsed_sec": 0.0, **step0_val_metrics}]
    final_beta_df = step0_beta_df
    cum_train_sec = 0.0
    cum_eval_sec = 0.0

    pbar = tqdm(enumerate(epoch_items, start=1), total=total_steps, desc=f"lambda={beta_lambda:g}")
    for step_idx, item in pbar:
        torch.cuda.synchronize()
        t_step0 = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)

        # 1. Single shared-prefix forward pass for all choice tasks of this respondent
        with torch.autocast("cuda", dtype=torch.bfloat16):
            resp_logits_list = kev_model.forward([item["enc"]], shared_prefix=True)[0]
        task_logits = torch.stack([z.to(dtype=torch.float64) for z in resp_logits_list], dim=0)
        torch.cuda.synchronize()
        t_after_fwd = time.perf_counter()

        # 2. Compute KL + respondent OLS beta loss on full-rank respondents, or KL-only when rank < 9
        if item["A_resp"] is not None:
            total_loss, kl_loss, beta_loss = compute_wu2025_loss(
                task_logits,
                item["y_resp"],
                item["A_resp"],
                beta_lambda=beta_lambda,
            )
            raw_beta_val = float(beta_loss.detach().cpu())
        else:
            log_probs = torch.log_softmax(task_logits, dim=-1)
            kl_loss = torch.nn.functional.kl_div(log_probs, item["y_resp"].view(-1, 2), reduction="batchmean")
            beta_loss = None
            total_loss = kl_loss
            raw_beta_val = float("nan")

        kl_val = float(kl_loss.detach().cpu())
        total_val = float(total_loss.detach().cpu())
        torch.cuda.synchronize()
        t_after_loss = time.perf_counter()

        # 3. Single backward pass, gradient clipping, AdamW step (`grad_accum=1`), and OneCycleLR step
        total_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        if not bool(torch.isfinite(grad_norm)):
            raise FloatingPointError(f"Non-finite gradient norm {float(grad_norm)} at step {step_idx}")
        grad_norm_val = float(grad_norm.detach().cpu())
        optimizer.step()
        if step_idx < total_steps:
            scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        # Free per-step autograd tensors immediately so no computation graph lingers across steps or evaluations
        del resp_logits_list, task_logits, total_loss, kl_loss, beta_loss
        torch.cuda.synchronize()
        t_after_bwd = time.perf_counter()

        fwd_ms = (t_after_fwd - t_step0) * 1000.0
        loss_ms = (t_after_loss - t_after_fwd) * 1000.0
        bwd_opt_ms = (t_after_bwd - t_after_loss) * 1000.0
        step_ms = (t_after_bwd - t_step0) * 1000.0
        cum_train_sec += t_after_bwd - t_step0

        step_rows.append(
            {
                "step": step_idx,
                "caseid": item["caseid"],
                "has_ols": item["A_resp"] is not None,
                "kl_loss": kl_val,
                "raw_beta_loss": raw_beta_val,
                "weighted_beta_loss": beta_lambda * raw_beta_val if item["A_resp"] is not None else 0.0,
                "total_loss": total_val,
                "grad_norm": grad_norm_val,
                "fwd_ms": fwd_ms,
                "loss_ms": loss_ms,
                "bwd_opt_ms": bwd_opt_ms,
                "step_ms": step_ms,
            }
        )

        # Update live progress bar with rolling 50-step statistical losses and step timings
        if step_idx % 25 == 0 or step_idx == total_steps:
            recent = step_rows[-50:]
            recent_betas = [r["raw_beta_loss"] for r in recent if r["has_ols"]]
            pbar.set_postfix(
                kl=f"{float(np.mean([r['kl_loss'] for r in recent])):.3f}",
                beta=f"{float(np.mean(recent_betas)):.3f}" if recent_betas else "n/a",
                step_ms=f"{float(np.mean([r['step_ms'] for r in recent])):.1f}",
                loss_ms=f"{float(np.mean([r['loss_ms'] for r in recent])):.2f}",
            )

        # 4. Periodically evaluate population OLS approximation and record eval timing alongside statistical metrics
        if step_idx % EPOCH_VAL_EVERY == 0 or step_idx == total_steps:
            v_metrics, final_beta_df = validate_wu2025_population_ols(
                kev_model,
                wu2025_pop_bundle,
                A_pop,
                y_pop,
                human_pop_betas,
            )
            cum_eval_sec += v_metrics["eval_wall_sec"]
            val_rows.append({"step": step_idx, "train_elapsed_sec": cum_train_sec, **v_metrics})
            tqdm.write(
                f"[lambda={beta_lambda:g} | step {step_idx:4d}/{total_steps}] "
                f"pop_kl={v_metrics['pop_kl_loss']:.4f} | "
                f"pop_beta_mse={v_metrics['pop_beta_mse']:.5f} | "
                f"MAPE={v_metrics['pop_beta_mape_pct']:.2f}% | "
                f"rho={v_metrics['spearman_rho']:.3f} | "
                f"train={cum_train_sec:.1f}s | "
                f"eval_gpu={v_metrics['eval_gpu_sec']:.2f}s | "
                f"eval_wall={v_metrics['eval_wall_sec']:.2f}s"
            )

    total_wall_sec = time.perf_counter() - t_wall0
    step_df = pd.DataFrame(step_rows)
    val_history_df = pd.DataFrame(val_rows).set_index("step")
    ols_steps = step_df[step_df["has_ols"]]

    # Summarize both statistical loss metrics and step / eval / wall-clock timing metrics
    summary = pd.Series(
        {
            "beta_lambda": beta_lambda,
            "train_kl_first200": float(step_df["kl_loss"].iloc[:200].mean()),
            "train_kl_last200": float(step_df["kl_loss"].iloc[-200:].mean()),
            "train_raw_beta_first200": float(ols_steps["raw_beta_loss"].iloc[:200].mean()),
            "train_raw_beta_last200": float(ols_steps["raw_beta_loss"].iloc[-200:].mean()),
            "train_weighted_beta_last200": float(ols_steps["weighted_beta_loss"].iloc[-200:].mean()),
            "train_total_first200": float(step_df["total_loss"].iloc[:200].mean()),
            "train_total_last200": float(step_df["total_loss"].iloc[-200:].mean()),
            "mean_grad_norm": float(step_df["grad_norm"].mean()),
            "mean_fwd_ms": float(step_df["fwd_ms"].mean()),
            "mean_loss_ms": float(step_df["loss_ms"].mean()),
            "mean_bwd_opt_ms": float(step_df["bwd_opt_ms"].mean()),
            "mean_step_ms": float(step_df["step_ms"].mean()),
            "p50_step_ms": float(step_df["step_ms"].median()),
            "p95_step_ms": float(step_df["step_ms"].quantile(0.95)),
            "mean_eval_wall_sec": float(val_history_df["eval_wall_sec"].iloc[1:].mean()),
            "mean_eval_gpu_sec": float(val_history_df["eval_gpu_sec"].iloc[1:].mean()),
            "train_only_sec": cum_train_sec,
            "eval_only_sec": cum_eval_sec,
            "total_wall_sec": total_wall_sec,
        },
        name=f"lambda_{beta_lambda:g}_epoch_summary",
    )
    display(summary.round(4).to_frame())
    display(val_history_df.round(4))

    return {
        "beta_lambda": beta_lambda,
        "summary": summary,
        "step_df": step_df,
        "val_history_df": val_history_df,
        "final_beta_df": final_beta_df,
    }


print("Defined run_wu2025_epoch(beta_lambda) with synchronized step and eval timing metrics")

Defined run_wu2025_epoch(beta_lambda) with synchronized step and eval timing metrics


In [32]:
# Run 1 full epoch with `beta_lambda = 0.0` (pure per-response KL baseline, identical to standard KL training)
lambda_runs[0.0] = run_wu2025_epoch(beta_lambda=0.0)

lambda=0:   0%|          | 0/1861 [00:00<?, ?it/s]

[lambda=0 | step  450/1861] pop_kl=0.5728 | pop_beta_mse=0.00763 | MAPE=36.55% | rho=1.000 | train=60.3s | eval_gpu=32.41s | eval_wall=33.63s
[lambda=0 | step  900/1861] pop_kl=0.5737 | pop_beta_mse=0.00472 | MAPE=31.61% | rho=0.976 | train=120.9s | eval_gpu=32.57s | eval_wall=33.79s
[lambda=0 | step 1350/1861] pop_kl=0.5416 | pop_beta_mse=0.00136 | MAPE=14.62% | rho=1.000 | train=182.9s | eval_gpu=33.54s | eval_wall=34.81s
[lambda=0 | step 1800/1861] pop_kl=0.5396 | pop_beta_mse=0.00149 | MAPE=14.11% | rho=0.976 | train=244.1s | eval_gpu=32.59s | eval_wall=33.80s
[lambda=0 | step 1861/1861] pop_kl=0.5397 | pop_beta_mse=0.00150 | MAPE=14.16% | rho=0.976 | train=252.2s | eval_gpu=32.74s | eval_wall=34.00s


,lambda_0_epoch_summary
beta_lambda,0.0000
train_kl_first200,0.6541
train_kl_last200,0.5420
train_raw_beta_first200,0.1780
train_raw_beta_last200,0.1566
train_weighted_beta_last200,0.0000
train_total_first200,0.6541
train_total_last200,0.5420
mean_grad_norm,5.3385
mean_fwd_ms,57.9001


,train_elapsed_sec,pop_kl_loss,pop_beta_mse,pop_beta_mae,pop_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,
0,0.0000,1.0699,0.0512,0.2093,114.1314,-0.2143,0.1943,32.9060,5.7037,34.2062,1.8399
450,60.3316,0.5728,0.0076,0.0707,36.5494,1.0000,0.0111,32.4106,6.4206,33.6339,1.8091
900,120.9402,0.5737,0.0047,0.0612,31.6060,0.9762,0.0109,32.5710,6.3667,33.7881,1.8174
1350,182.8532,0.5416,0.0014,0.0302,14.6208,1.0000,0.0125,33.5411,6.5803,34.8091,1.8724
1800,244.1210,0.5396,0.0015,0.0313,14.1081,0.9762,0.0111,32.5929,6.6697,33.8029,1.8182
1861,252.2059,0.5397,0.0015,0.0314,14.1591,0.9762,0.0109,32.7409,7.1260,34.0027,1.8290


In [33]:
# Run 1 full epoch with `beta_lambda = 1.0` (mild per-respondent OLS beta weight)
lambda_runs[1.0] = run_wu2025_epoch(beta_lambda=1.0)

lambda=1:   0%|          | 0/1861 [00:00<?, ?it/s]

[lambda=1 | step  450/1861] pop_kl=0.5645 | pop_beta_mse=0.00488 | MAPE=27.44% | rho=0.976 | train=61.4s | eval_gpu=32.50s | eval_wall=33.72s
[lambda=1 | step  900/1861] pop_kl=0.5784 | pop_beta_mse=0.00496 | MAPE=31.37% | rho=0.976 | train=122.6s | eval_gpu=32.61s | eval_wall=33.82s
[lambda=1 | step 1350/1861] pop_kl=0.5412 | pop_beta_mse=0.00141 | MAPE=16.69% | rho=1.000 | train=183.2s | eval_gpu=33.52s | eval_wall=34.77s
[lambda=1 | step 1800/1861] pop_kl=0.5363 | pop_beta_mse=0.00158 | MAPE=15.36% | rho=0.976 | train=244.4s | eval_gpu=33.44s | eval_wall=34.69s
[lambda=1 | step 1861/1861] pop_kl=0.5362 | pop_beta_mse=0.00159 | MAPE=15.36% | rho=0.976 | train=252.8s | eval_gpu=32.01s | eval_wall=33.24s


,lambda_1_epoch_summary
beta_lambda,1.0000
train_kl_first200,0.6501
train_kl_last200,0.5389
train_raw_beta_first200,0.1765
train_raw_beta_last200,0.1543
train_weighted_beta_last200,0.1543
train_total_first200,0.8162
train_total_last200,0.6770
mean_grad_norm,6.6229
mean_fwd_ms,58.0945


,train_elapsed_sec,pop_kl_loss,pop_beta_mse,pop_beta_mae,pop_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,
0,0.0000,1.0699,0.0512,0.2093,114.1314,-0.2143,0.1943,32.9060,5.7037,34.2062,1.8399
450,61.3939,0.5645,0.0049,0.0550,27.4422,0.9762,0.0110,32.5004,6.4105,33.7219,1.8139
900,122.6368,0.5784,0.0050,0.0616,31.3703,0.9762,0.0114,32.6050,6.4620,33.8166,1.8190
1350,183.1716,0.5412,0.0014,0.0325,16.6872,1.0000,0.0111,33.5211,6.5315,34.7656,1.8700
1800,244.4370,0.5363,0.0016,0.0330,15.3647,0.9762,0.0122,33.4432,6.5978,34.6942,1.8662
1861,252.8371,0.5362,0.0016,0.0331,15.3637,0.9762,0.0111,32.0102,6.6028,33.2365,1.7878


In [34]:
# Run 1 full epoch with `beta_lambda = 5.0` (moderate per-respondent OLS beta weight)
lambda_runs[5.0] = run_wu2025_epoch(beta_lambda=5.0)

lambda=5:   0%|          | 0/1861 [00:00<?, ?it/s]

[lambda=5 | step  450/1861] pop_kl=0.5745 | pop_beta_mse=0.00659 | MAPE=36.94% | rho=0.976 | train=60.8s | eval_gpu=32.80s | eval_wall=34.02s
[lambda=5 | step  900/1861] pop_kl=0.6083 | pop_beta_mse=0.00828 | MAPE=38.37% | rho=0.976 | train=121.1s | eval_gpu=32.76s | eval_wall=33.97s
[lambda=5 | step 1350/1861] pop_kl=0.5379 | pop_beta_mse=0.00090 | MAPE=13.38% | rho=1.000 | train=181.5s | eval_gpu=33.44s | eval_wall=34.68s
[lambda=5 | step 1800/1861] pop_kl=0.5349 | pop_beta_mse=0.00157 | MAPE=17.56% | rho=0.976 | train=244.1s | eval_gpu=33.12s | eval_wall=34.35s
[lambda=5 | step 1861/1861] pop_kl=0.5349 | pop_beta_mse=0.00157 | MAPE=17.58% | rho=0.976 | train=252.4s | eval_gpu=32.95s | eval_wall=34.18s


,lambda_5_epoch_summary
beta_lambda,5.0000
train_kl_first200,0.6670
train_kl_last200,0.5292
train_raw_beta_first200,0.1771
train_raw_beta_last200,0.1529
train_weighted_beta_last200,0.7645
train_total_first200,1.4957
train_total_last200,1.2107
mean_grad_norm,15.4702
mean_fwd_ms,57.9906


,train_elapsed_sec,pop_kl_loss,pop_beta_mse,pop_beta_mae,pop_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,
0,0.0000,1.0699,0.0512,0.2093,114.1314,-0.2143,0.1943,32.9060,5.7037,34.2062,1.8399
450,60.7521,0.5745,0.0066,0.0689,36.9360,0.9762,0.0112,32.8031,6.4795,34.0164,1.8297
900,121.1340,0.6083,0.0083,0.0771,38.3687,0.9762,0.0109,32.7557,6.5398,33.9721,1.8273
1350,181.4674,0.5379,0.0009,0.0240,13.3777,1.0000,0.0114,33.4426,6.5613,34.6784,1.8653
1800,244.0588,0.5349,0.0016,0.0346,17.5619,0.9762,0.0115,33.1226,6.7125,34.3510,1.8477
1861,252.3670,0.5349,0.0016,0.0347,17.5833,0.9762,0.0112,32.9501,6.6108,34.1805,1.8385


In [35]:
# Run 1 full epoch with `beta_lambda = 20.0` (strong per-respondent OLS beta weight)
lambda_runs[20.0] = run_wu2025_epoch(beta_lambda=20.0)

lambda=20:   0%|          | 0/1861 [00:00<?, ?it/s]

[lambda=20 | step  450/1861] pop_kl=0.5763 | pop_beta_mse=0.00723 | MAPE=31.93% | rho=1.000 | train=60.3s | eval_gpu=32.11s | eval_wall=33.33s
[lambda=20 | step  900/1861] pop_kl=0.6569 | pop_beta_mse=0.01107 | MAPE=44.71% | rho=0.976 | train=119.2s | eval_gpu=32.11s | eval_wall=33.31s
[lambda=20 | step 1350/1861] pop_kl=0.5455 | pop_beta_mse=0.00185 | MAPE=19.47% | rho=0.976 | train=179.3s | eval_gpu=33.21s | eval_wall=34.46s
[lambda=20 | step 1800/1861] pop_kl=0.5403 | pop_beta_mse=0.00207 | MAPE=21.26% | rho=0.976 | train=239.2s | eval_gpu=32.46s | eval_wall=33.68s
[lambda=20 | step 1861/1861] pop_kl=0.5403 | pop_beta_mse=0.00207 | MAPE=21.25% | rho=0.976 | train=247.3s | eval_gpu=32.52s | eval_wall=33.75s


,lambda_20_epoch_summary
beta_lambda,20.0000
train_kl_first200,0.6885
train_kl_last200,0.5332
train_raw_beta_first200,0.1834
train_raw_beta_last200,0.1538
train_weighted_beta_last200,3.0753
train_total_first200,4.1187
train_total_last200,3.2688
mean_grad_norm,49.9794
mean_fwd_ms,56.7446


,train_elapsed_sec,pop_kl_loss,pop_beta_mse,pop_beta_mae,pop_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,
0,0.0000,1.0699,0.0512,0.2093,114.1314,-0.2143,0.1943,32.9060,5.7037,34.2062,1.8399
450,60.3437,0.5763,0.0072,0.0653,31.9306,1.0000,0.0113,32.1086,6.6573,33.3259,1.7926
900,119.1826,0.6569,0.0111,0.0893,44.7100,0.9762,0.0110,32.1077,6.5013,33.3123,1.7918
1350,179.2873,0.5455,0.0019,0.0351,19.4671,0.9762,0.0111,33.2128,6.6891,34.4614,1.8537
1800,239.1923,0.5403,0.0021,0.0403,21.2577,0.9762,0.0111,32.4565,6.8805,33.6758,1.8114
1861,247.2615,0.5403,0.0021,0.0403,21.2507,0.9762,0.0110,32.5163,6.6118,33.7486,1.8153


In [36]:
# Compare statistical metrics and step / eval / epoch timing metrics side-by-side across completed `beta_lambda` runs
if not lambda_runs:
    raise ValueError("Run at least one `beta_lambda` epoch cell first so `lambda_runs` is populated")
ran_lambdas = sorted(lambda_runs.keys())

compare_rows = [
    {
        "run": "step_0 (untrained)",
        "beta_lambda": float("nan"),
        "train_kl_last200": float("nan"),
        "train_raw_beta_last200": float("nan"),
        "pop_kl_loss": step0_val_metrics["pop_kl_loss"],
        "pop_beta_mse": step0_val_metrics["pop_beta_mse"],
        "pop_beta_mae": step0_val_metrics["pop_beta_mae"],
        "pop_beta_mape_pct": step0_val_metrics["pop_beta_mape_pct"],
        "spearman_rho": step0_val_metrics["spearman_rho"],
        "mean_loss_ms": float("nan"),
        "mean_step_ms": float("nan"),
        "mean_eval_wall_sec": step0_val_metrics["eval_wall_sec"],
        "train_only_sec": 0.0,
        "eval_only_sec": step0_val_metrics["eval_wall_sec"],
        "total_wall_sec": step0_val_metrics["eval_wall_sec"],
    }
]
for lam in ran_lambdas:
    run = lambda_runs[lam]
    s = run["summary"]
    final_val = run["val_history_df"].iloc[-1]
    compare_rows.append(
        {
            "run": f"lambda={lam:g}",
            "beta_lambda": lam,
            "train_kl_last200": float(s["train_kl_last200"]),
            "train_raw_beta_last200": float(s["train_raw_beta_last200"]),
            "pop_kl_loss": float(final_val["pop_kl_loss"]),
            "pop_beta_mse": float(final_val["pop_beta_mse"]),
            "pop_beta_mae": float(final_val["pop_beta_mae"]),
            "pop_beta_mape_pct": float(final_val["pop_beta_mape_pct"]),
            "spearman_rho": float(final_val["spearman_rho"]),
            "mean_loss_ms": float(s["mean_loss_ms"]),
            "mean_step_ms": float(s["mean_step_ms"]),
            "mean_eval_wall_sec": float(s["mean_eval_wall_sec"]),
            "train_only_sec": float(s["train_only_sec"]),
            "eval_only_sec": float(s["eval_only_sec"]),
            "total_wall_sec": float(s["total_wall_sec"]),
        }
    )

comparison_df = pd.DataFrame(compare_rows).set_index("run")
display(comparison_df.round(4))

# Compare the 8 learned population OLS betas against the ground-truth human population OLS betas
pop_betas_compare_df = pd.DataFrame(
    {
        "human_pop_beta": step0_beta_df["human_pop_beta"],
        "step_0": step0_beta_df["model_pop_beta"],
        **{f"lambda={lam:g}": lambda_runs[lam]["final_beta_df"]["model_pop_beta"] for lam in ran_lambdas},
    }
)
display(pop_betas_compare_df.round(4))

,beta_lambda,train_kl_last200,train_raw_beta_last200,pop_kl_loss,pop_beta_mse,pop_beta_mae,pop_beta_mape_pct,spearman_rho,mean_loss_ms,mean_step_ms,mean_eval_wall_sec,train_only_sec,eval_only_sec,total_wall_sec
run,,,,,,,,,,,,,,
step_0 (untrained),NaN,NaN,NaN,1.0699,0.0512,0.2093,114.1314,-0.2143,NaN,NaN,34.2062,0.0000,34.2062,34.2062
lambda=0,0.0,0.5420,0.1566,0.5397,0.0015,0.0314,14.1591,0.9762,0.2834,135.5217,34.0073,252.2059,170.0367,423.2422
lambda=1,1.0,0.5389,0.1543,0.5362,0.0016,0.0331,15.3637,0.9762,0.2957,135.8609,34.0470,252.8371,170.2348,424.0477
lambda=5,5.0,0.5292,0.1529,0.5349,0.0016,0.0347,17.5833,0.9762,0.2929,135.6083,34.2397,252.3670,171.1984,424.5080
lambda=20,20.0,0.5332,0.1538,0.5403,0.0021,0.0403,21.2507,0.9762,0.2828,132.8649,33.7048,247.2615,168.5240,416.7107


,human_pop_beta,step_0,lambda=0,lambda=1,lambda=5,lambda=20
term,,,,,,
Monthly bill: $110 vs $105,-0.0969,-0.0068,-0.0979,-0.1105,-0.1177,-0.1145
Monthly bill: $115 vs $105,-0.1779,-0.0122,-0.1999,-0.2029,-0.2046,-0.1959
Monthly bill: $120 vs $105,-0.2894,-0.0171,-0.3386,-0.3497,-0.3462,-0.3596
Monthly bill: $125 vs $105,-0.3832,-0.0248,-0.4436,-0.4503,-0.4478,-0.4479
Percentage of renewables: 30% vs 10%,0.0973,-0.0075,0.1142,0.1128,0.1228,0.1353
Percentage of renewables: 50% vs 10%,0.1706,-0.0149,0.2087,0.2005,0.2003,0.2118
Percentage of renewables: 100% vs 10%,0.2421,-0.0398,0.3033,0.2932,0.2922,0.3012
Locally produced renewables: Yes vs No,0.1042,-0.1109,0.1066,0.1063,0.1079,0.1178
